In [6]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

def _load_train(paths):
    for p in paths:
        if os.path.exists(p):
            return pd.read_csv(p)
    raise FileNotFoundError(f"Không tìm thấy file train. Đã thử: {paths}")

def evaluate_train_test(estimator, X, y, test_size=0.2, random_state=42):
    X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=test_size, stratify=y, random_state=random_state)
    clf = estimator
    clf.fit(X_tr, y_tr)
    y_pred = clf.predict(X_val)

    # try to get score/proba for ROC AUC
    y_score = None
    if hasattr(clf, "predict_proba"):
        y_score = clf.predict_proba(X_val)[:, 1]
    elif hasattr(clf, "decision_function"):
        y_score = clf.decision_function(X_val)
    else:
        y_score = y_pred  # fallback (will often give poor ROC AUC)

    # safe roc computation
    try:
        roc = roc_auc_score(y_val, y_score)
    except Exception:
        roc = np.nan

    return {
        "accuracy": accuracy_score(y_val, y_pred),
        "f1": f1_score(y_val, y_pred),
        "roc_auc": roc
    }

def evaluate_kfold(estimator, X, y, n_splits=5, random_state=42, n_jobs=-1):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    # cross-validated label preds for accuracy & f1
    y_pred_cv = cross_val_predict(estimator, X, y, cv=cv, n_jobs=n_jobs)

    # try to get scores for roc auc using predict_proba or decision_function
    y_score_cv = None
    if hasattr(estimator, "predict_proba"):
        try:
            y_score_cv = cross_val_predict(estimator, X, y, cv=cv, method="predict_proba", n_jobs=n_jobs)[:, 1]
        except Exception:
            y_score_cv = None
    if y_score_cv is None and hasattr(estimator, "decision_function"):
        try:
            y_score_cv = cross_val_predict(estimator, X, y, cv=cv, method="decision_function", n_jobs=n_jobs)
        except Exception:
            y_score_cv = None
    if y_score_cv is None:
        # fallback to labels (not recommended)
        y_score_cv = y_pred_cv

    try:
        roc = roc_auc_score(y, y_score_cv)
    except Exception:
        roc = np.nan

    return {
        "accuracy": accuracy_score(y, y_pred_cv),
        "f1": f1_score(y, y_pred_cv),
        "roc_auc": roc
    }

if __name__ == "__main__":
    # thử nhiều tên file train (tùy cấu trúc repo của bạn)
    possible_paths = [
        os.path.join("..", "data", "titanic_FINAL_INPUT_train.csv"),
        os.path.join("..", "data", "titanic_train.csv"),
        "titanic_train.csv",
        os.path.join("..", "Titanic - Machine Learning from Disaster", "titanic_train.csv"),
    ]
    train = _load_train(possible_paths)

    # chuẩn hoá cột mục tiêu / loại bỏ PassengerId nếu có
    if "Survived" not in train.columns:
        raise RuntimeError("File train không chứa cột 'Survived'")

    X = train.drop([c for c in ["PassengerId", "Survived"] if c in train.columns], axis=1)
    y = train["Survived"]

    models = {
        "SVM": SVC(random_state=42, probability=True),
        "RandomForest": RandomForestClassifier(random_state=42),
        "KNN": KNeighborsClassifier()
    }

    # đánh giá theo Train/Test Split
    tt_rows = []
    for name, model in models.items():
        metrics = evaluate_train_test(model, X, y, test_size=0.2, random_state=42)
        tt_rows.append({"model": name, **metrics})
    df_tt = pd.DataFrame(tt_rows).set_index("model")

    # đánh giá theo K-Fold CV
    kf_rows = []
    for name, model in models.items():
        metrics = evaluate_kfold(model, X, y, n_splits=5, random_state=42, n_jobs=-1)
        kf_rows.append({"model": name, **metrics})
    df_kf = pd.DataFrame(kf_rows).set_index("model")

    # hiển thị 2 bảng riêng biệt và lưu CSV
    print("\n--- Train/Test Split Evaluation ---")
    print(df_tt)
    df_tt.to_csv("evaluation_train_test.csv", index=True)

    print("\n--- K-Fold Cross-Validation Evaluation ---")
    print(df_kf)
    df_kf.to_csv("evaluation_kfold.csv", index=True)

    print("\nSaved: evaluation_train_test.csv, evaluation_kfold.csv")    



--- Train/Test Split Evaluation ---
              accuracy        f1   roc_auc
model                                     
SVM           0.843575  0.787879  0.835046
RandomForest  0.748603  0.676259  0.816667
KNN           0.821229  0.750000  0.840250

--- K-Fold Cross-Validation Evaluation ---
              accuracy        f1   roc_auc
model                                     
SVM           0.835017  0.774194  0.853623
RandomForest  0.812570  0.748872  0.872735
KNN           0.818182  0.748447  0.863801

Saved: evaluation_train_test.csv, evaluation_kfold.csv
